# 99: Publication figures (APSRC extended abstract)

Four figures for the APSRC extended abstract draft, each an **extension or
recombination of an existing figure** from notebooks 03/05/06 — nothing here
re-derives a number those notebooks don't already produce.

| # | Figure | Built from |
|---|---|---|
| 1 | Daily plot, compliant vs non-compliant, side by side | `05_site_explorer.ex.plot_operational` x2, stitched |
| 2 | Volt-VAr response by class, 4 panels | `05_site_explorer.ex.plot_site_voltvar_curve` x4, stitched |
| 3 | Site-level results: Volt-Watt / Volt-VAr / combined | new — draft, see note in that section |
| 4 | Curtailment per mode + 2 Lorenz curves | `06_curtailment` totals + `se_plots.plot_curtailment_lorenz` x2, combined |

**Site IDs are left as placeholders** (`SITE_...= None`) throughout — the
notebook prints candidate sites where it can, but you choose and fill in the
final `site_alias` values before running each figure's cell.

Figures 3 and 4 need the same fleet-scale frames notebook 06 builds
(`method_b`, `vw_curt`, `cohort_sites`, `fleet_generation`) — this notebook
recomputes them the same way, so expect the same runtime as 06 the first time
you run it.


In [43]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents)
                  if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()), None)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

from solar_edge.config import se_config as C
from solar_edge.lib import se_store, se_contract as contract, se_params
from solar_edge.lib import se_explore as ex
from solar_edge.lib import se_curtailment as cu
from solar_edge.lib import se_plots as plots
from solar_edge.lib import se_adverse as adv
from solar_edge.lib import se_sign as sign

con = se_store.connect()
config, params = se_params.CONFIG, se_params.PARAMS

FIG_DIR = C.ARTEFACT_DIR / "publication_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures will be saved to: {FIG_DIR}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Figures will be saved to: C:\Users\z3553082\OneDrive - UNSW\Documents\GitHub\CICCADA\solar_edge\artefacts\publication_figures


## 0. Candidate sites

Not required, but a starting point: `classify_adverse_sites` already sorts
the fleet into the four buckets Figure 2 wants (`not_adverse` covers both
"responsive" and "not responsive" -- split it further below by response
magnitude), and `site_days_available` ranks a given site's days by how much
time they spend in the Volt-VAr band, which is what you want for Figure 1's
daily plot.

Nothing here auto-selects a site — it only narrows the field so you can pick
one you trust from `03_conformance` / `05_site_explorer` and paste its
`site_alias` into the placeholders in each figure's section below.


In [ ]:
classified = adv.classify_adverse_sites(con, config)
display(classified.adverse_class.value_counts())

print("\n-- responsive candidates (not_adverse, high ratio_to_required) --")
display(classified[classified.adverse_class == "not_adverse"]
        .sort_values("ratio_to_required", ascending=False).head(8))

print("\n-- not-responsive candidates (not_adverse, low ratio_to_required) --")
display(classified[classified.adverse_class == "not_adverse"]
        .sort_values("ratio_to_required", ascending=True).head(8))

print("\n-- polarity-suspect candidates (\"potentially sign-wrong\") --")
display(classified[classified.adverse_class == "polarity_suspect"]
        .sort_values("ratio_to_required", ascending=False).head(8))

print("\n-- genuinely-adverse candidates (\"genuinely inverted\") --")
display(classified[classified.adverse_class == "genuinely_adverse"]
        .sort_values("mad_magnitude_kvar", ascending=False).head(8))


## 0b. Fleet-scale frames (needed for Figures 3 and 4)

Same calls as `06_curtailment.ipynb` — reproduced here rather than imported
from a CSV, because `curtailment_concentration()` stores the full per-site
share vector in `.attrs`, which does not survive a CSV round-trip (see
`se_curtailment.method_comparison`'s guard for the same reason). The Lorenz
curves in Figure 4 need it, so these are recomputed live.

If you've already run 06 in this kernel session, you can skip this cell and
reuse its `method_b` / `vw_curt` / `cohort_sites` / `fleet_generation`
variables directly.


In [ ]:
adverse = classified  # same frame, alias to match 06's naming

method_b = cu.method_b_site_year(con, config, params, adverse=adverse)
vw_curt = cu.voltwatt_curtailment_site_year(con, config, params)

cohort_sites = cu.cohort_site_count(con, config)
fleet_generation = cu.fleet_potential_generation(con, config)

concentration = cu.curtailment_concentration(method_b, config, mode="voltvar")
vw_concentration = cu.curtailment_concentration(vw_curt, config, mode="voltwatt")

print(f"Volt-VAr: {concentration.attrs.get('total_kWh', 0):,.1f} kWh across "
      f"{concentration.attrs.get('n_affected_sites', 0):,} affected sites")
print(f"Volt-Watt: {vw_concentration.attrs.get('total_kWh', 0):,.1f} kWh across "
      f"{vw_concentration.attrs.get('n_affected_sites', 0):,} affected sites")


Volt-VAr: 1,199.9 kWh across 484 affected sites
Volt-Watt: 5,547.8 kWh across 329 affected sites


## Figure 1 — Daily plot, compliant site vs non-compliant site

Reuses `ex.plot_operational` (the notebook 05 "daily operational plot")
completely unchanged, once per site, then stitches the two PNGs side by
side — left panel compliant, right panel non-compliant/adverse.

**Site IDs are redacted on the figure itself.** `plot_operational`'s
`site_alias` argument is only ever used for the title text (never for the
underlying query, which runs on the `df_day` you already pulled), so the
helper below passes an anonymised label (`"Site A"` / `"Site B"`) into that
argument instead of the real `site_alias`. The real ID still appears in the
filename and in this notebook's own cell output, for your own reference —
only the rendered PNG is redacted.


In [ ]:
# --- fill these in from the candidates above, or from 03/05 directly ---
SITE_COMPLIANT = "AUS976"       # e.g. "AUS068"
SITE_NONCOMPLIANT = "AUS978"    # e.g. "AUS351"
DAY_COMPLIANT = "2025-04-16"        # optional -- e.g. "2025-04-16"; None picks the top day
DAY_NONCOMPLIANT = "2025-04-16"

assert SITE_COMPLIANT and SITE_NONCOMPLIANT, (
    "Set SITE_COMPLIANT and SITE_NONCOMPLIANT above before running this cell."
)


In [ ]:
def _pick_day(site, day):
    avail = ex.site_days_available(con, site)
    if avail.empty:
        raise ValueError(
            f"{site!r}: site_days_available() returned no rows -- either the "
            "site_alias is wrong (check case/spelling against the candidates "
            "cell above) or it has no day with enough Volt-VAr-band intervals "
            "(site_days_available's default min_intervals=100)."
        )
    if day is not None:
        if day not in set(avail.day_aest.astype(str).str[:10]):
            raise ValueError(
                f"{site!r} has no data on {day!r}. Days available (ranked by "
                f"time in the Volt-VAr band), most first:\n"
                f"{avail.day_aest.astype(str).str[:10].head(10).tolist()}\n"
                "Pass one of these, or leave the DAY_* placeholder as None to "
                "use the top-ranked day automatically."
            )
        return day
    return str(avail.day_aest.iloc[0])[:10]

def _boost_legends(fig, scale=2.0):
    """Rebuild every legend in ``fig`` at ``scale`` x its original fontsize.

    Recreated from the existing Legend's own handles/labels (rather than
    ``ax.get_legend_handles_labels()``) because several panels build their
    legend from proxy artists (``Patch``/``Line2D``) passed via
    ``legend(handles=[...])`` that are never added to the axes as real
    artists -- ``get_legend_handles_labels()`` would not find them.
    """
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is None:
            continue
        handles = leg.legendHandles
        labels = [t.get_text() for t in leg.get_texts()]
        base_fontsize = leg.get_texts()[0].get_fontsize() if leg.get_texts() else 7
        loc = leg._loc
        framealpha = leg.get_frame().get_alpha()
        leg.remove()
        ax.legend(handles=handles, labels=labels, fontsize=base_fontsize * scale,
                  loc=loc, framealpha=framealpha)

def _drop_title_line(fig, line_idx=1):
    """Remove one line from the figure's suptitle (0-indexed)."""
    if fig._suptitle is None:
        return
    lines = fig._suptitle.get_text().split("\n")
    if len(lines) > line_idx:
        del lines[line_idx]
    fig._suptitle.set_text("\n".join(lines))

def daily_plot_png(site, day, anon_label, orientation="stored"):
    day = _pick_day(site, day)
    frame = ex.site_day(con, site, day)
    if frame.empty:
        raise ValueError(
            f"ex.site_day(con, {site!r}, {day!r}) returned no rows, even though "
            "site_days_available() listed this day -- check the day string is "
            "exactly 'YYYY-MM-DD' with no time component."
        )
    # anon_label, not site, goes into the plot -- plot_operational only uses this
    # argument for the title text, never for the query, so the real site_alias
    # never reaches the figure.
    fig = ex.plot_operational(frame, anon_label, float(frame.s_99.iloc[0]), day,
                              orientation=orientation)

    # Publication tweaks -- post-process rather than editing plot_operational,
    # since that function is shared with 05_site_explorer and these are
    # publication-figure-specific preferences, not a fix to the library plot.
    _drop_title_line(fig, line_idx=1)                    # drop "REACTIVE SIGN: ..."
    fig.subplots_adjust(top=0.94)                         # less gap under the title
    fig._suptitle.set_y(0.975)                            # title closer to the axes
    _boost_legends(fig, scale=2.0)                        # 2x legend box on every panel

    path = FIG_DIR / f"daily_{anon_label.replace(' ', '_')}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"{anon_label} = {site} ({day})  [real ID kept out of the figure/filename]")
    return path

def side_by_side(path_left, path_right, out_path, gap_px=24):
    im_l, im_r = Image.open(path_left), Image.open(path_right)
    h = max(im_l.height, im_r.height)
    combined = Image.new("RGB", (im_l.width + gap_px + im_r.width, h), "white")
    combined.paste(im_l, (0, 0))
    combined.paste(im_r, (im_l.width + gap_px, 0))
    combined.save(out_path)
    return combined


In [ ]:
left_png = daily_plot_png(SITE_COMPLIANT, DAY_COMPLIANT, "Site A")
right_png = daily_plot_png(SITE_NONCOMPLIANT, DAY_NONCOMPLIANT, "Site B")

fig1_path = FIG_DIR / "fig1_daily_compliant_vs_noncompliant.png"
combined = side_by_side(left_png, right_png, fig1_path)
print(f"-> {fig1_path}")
combined


## Figure 2 — Volt-VAr response by class (4 panels)

Reuses `ex.plot_voltvar_scatter` (the notebook 05 whole-of-store Q-vs-V
scatter, coloured by the same D9 verdict that produced the fleet numbers —
not the median-curve view from earlier in 05), once per site, then stitches
the four PNGs into a 2x2 grid: responsive / not responsive / polarity-suspect
("potentially sign-wrong") / genuinely adverse ("genuinely inverted").

Mirrors this cell from `05_site_explorer.ipynb`:

```python
cats_all = ex.site_interval_categories(con, SITE, config=config)
profile = ex.site_profile(con, SITE, config)
display(ex.plot_voltvar_scatter(
    cats_all, SITE, float(profile.s_99),
    period_label=f"{cats_all.ts_aest.min():%Y-%m-%d} to {cats_all.ts_aest.max():%Y-%m-%d}",
    config=config))
```

**Site IDs are redacted on the figure itself** — `plot_voltvar_scatter`'s
`site_alias` argument is only used for the title text and the printed
category breakdown, never for the query, so the helper below passes an
anonymised label (`"Site A"`-`"Site D"`) instead of the real `site_alias`.
This scan runs over the whole store per site (not just one day), so expect
each panel to take longer than Figure 1's daily plot.


In [ ]:
# --- fill these in from the "0. Candidate sites" cell above ---
SITE_RESPONSIVE = "AUS976"          # not_adverse, high ratio_to_required
SITE_NOT_RESPONSIVE = "AUS978"      # not_adverse, low ratio_to_required
SITE_POLARITY_SUSPECT = "AUS1426"    # adverse_class == "polarity_suspect"
SITE_GENUINELY_ADVERSE = "AUS203"   # adverse_class == "genuinely_adverse"

assert all([SITE_RESPONSIVE, SITE_NOT_RESPONSIVE, SITE_POLARITY_SUSPECT,
           SITE_GENUINELY_ADVERSE]), (
    "Set all four SITE_* placeholders above before running this cell."
)


In [ ]:
def _check_site_exists(site):
    n = con.execute(
        "SELECT count(*) FROM se_site WHERE site_alias = ?", [site]
    ).fetchone()[0]
    if n == 0:
        raise ValueError(
            f"{site!r} is not a site_alias in se_site -- check it against the "
            "'0. Candidate sites' cell above (it's easy to paste a DataFrame "
            "index, a numpy value, or a value with stray whitespace/quotes "
            "instead of the plain site_alias string)."
        )

def _drop_axes_title_line(ax, line_idx=1, loc="left", **title_kwargs):
    """Remove one line from a single Axes' title (0-indexed)."""
    lines = ax.get_title(loc=loc).split("\n")
    if len(lines) > line_idx:
        del lines[line_idx]
    ax.set_title("\n".join(lines), loc=loc, **title_kwargs)

def voltvar_scatter_png(site, anon_label):
    """Whole-store Q-vs-V scatter for one site, coloured by D9 verdict.

    Mirrors 05_site_explorer's ``cats_all = ex.site_interval_categories(...)``
    / ``ex.plot_voltvar_scatter(...)`` cell. ``anon_label`` (not ``site``) is
    passed as plot_voltvar_scatter's ``site_alias`` argument -- that argument
    only feeds the title text and the printed category breakdown, never the
    query, so the real site_alias never reaches the figure or its filename.
    """
    _check_site_exists(site)
    cats_all = ex.site_interval_categories(con, site, config=config)
    if cats_all.empty:
        raise ValueError(
            f"ex.site_interval_categories(con, {site!r}) returned no rows -- "
            "this site exists but has no scored intervals. Try a different "
            "site from the candidates cell."
        )
    profile = ex.site_profile(con, site, config)
    fig = ex.plot_voltvar_scatter(
        cats_all, anon_label, float(profile.s_99),
        period_label=f"{cats_all.ts_aest.min():%Y-%m-%d} to {cats_all.ts_aest.max():%Y-%m-%d}",
        config=config,
    )

    # Publication tweaks -- post-process rather than editing plot_voltvar_scatter,
    # since that function is shared with 05_site_explorer.
    _drop_axes_title_line(fig.axes[0], line_idx=1, loc="left",
                          fontsize=10, fontweight="bold")   # drop the "coloured by D9 verdict" line
    _boost_legends(fig, scale=2.0)                          # 2x legend box

    path = FIG_DIR / f"voltvar_scatter_{anon_label.replace(' ', '_')}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"{anon_label} = {site}  [real ID kept out of the figure/filename]")
    return path

def grid_2x2(paths, out_path, gap_px=20):
    ims = [Image.open(p) for p in paths]
    w = max(im.width for im in ims)
    h = max(im.height for im in ims)
    combined = Image.new("RGB", (2 * w + gap_px, 2 * h + gap_px), "white")
    positions = [(0, 0), (w + gap_px, 0), (0, h + gap_px), (w + gap_px, h + gap_px)]
    for im, pos in zip(ims, positions):
        combined.paste(im, pos)
    combined.save(out_path)
    return combined


In [ ]:
panels = [
    voltvar_scatter_png(SITE_RESPONSIVE, "Site A"),
    voltvar_scatter_png(SITE_NOT_RESPONSIVE, "Site B"),
    voltvar_scatter_png(SITE_POLARITY_SUSPECT, "Site C"),
    voltvar_scatter_png(SITE_GENUINELY_ADVERSE, "Site D"),
]

fig2_path = FIG_DIR / "fig2_voltvar_response_by_class.png"
combined = grid_2x2(panels, fig2_path)
print(f"-> {fig2_path}")
combined


## Figure 3 — Site-level results: Volt-Watt, Volt-VAr, and combined

**Draft.** No figure like this exists yet in 03/05/06 — this is a first cut
built directly from the same per-site frames Figure 4 uses (`method_b` for
Volt-VAr, `vw_curt` for Volt-Watt), so you have something to start from
rather than nothing. Likely worth reworking before submission.

Shows the `TOP_N` sites by combined curtailed energy, stacked by mode.
**Site IDs are redacted on the figure** — the y-axis is labelled by rank
(`Site 1`, `Site 2`, ...) rather than `site_alias`; the real IDs are kept in
`site_level` (the DataFrame, printed separately below the chart) for your
own reference only.

**Not included in the submission** — kept here for later, per request. Skip this section when assembling the abstract's figure set.


In [ ]:
TOP_N = 25  # placeholder -- adjust to taste

h = config.interval_h
vvar_kwh = (method_b.groupby("site_alias").attributed_kw_sum.sum() * h).rename("voltvar_kWh")
vwatt_kwh = (vw_curt.groupby("site_alias").curtailed_kw_sum.sum() * h).rename("voltwatt_kWh")

site_level = pd.concat([vvar_kwh, vwatt_kwh], axis=1).fillna(0.0)
site_level["combined_kWh"] = site_level.voltvar_kWh + site_level.voltwatt_kWh
site_level = site_level.sort_values("combined_kWh", ascending=False).head(TOP_N)
site_level = site_level.reset_index().rename(columns={"index": "site_alias"})
# Rank-based label for the FIGURE only -- the real site_alias stays in this
# DataFrame (printed below) for your own reference, never on the chart.
site_level["anon_label"] = [f"Site {i+1}" for i in range(len(site_level))]

fig, ax = plt.subplots(figsize=(10, max(4, 0.32 * len(site_level))), dpi=130)
y = np.arange(len(site_level))
ax.barh(y, site_level.voltvar_kWh, color="#7c3aed", label="Volt-VAr")
ax.barh(y, site_level.voltwatt_kWh, left=site_level.voltvar_kWh,
       color="#4709b2", label="Volt-Watt")
ax.set_yticks(y)
ax.set_yticklabels(site_level.anon_label, fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("Estimated curtailment (kWh)")
ax.set_title(f"Top {TOP_N} sites by combined estimated curtailment\n"
             "(Volt-VAr = Method B attributed displacement; Volt-Watt = shed energy)",
             fontsize=10)
ax.legend(fontsize=8, frameon=False)
ax.grid(color="#ebebeb", lw=0.5, axis="x")
ax.set_axisbelow(True)
fig.tight_layout()

fig3_path = FIG_DIR / "fig3_site_level_combined_DRAFT.png"
fig.savefig(fig3_path, dpi=150, bbox_inches="tight")
print(f"-> {fig3_path}  (DRAFT)")
display(fig)
print("\nRank -> real site_alias lookup (NOT for the figure/paper):")
display(site_level[["anon_label", "site_alias", "voltvar_kWh", "voltwatt_kWh", "combined_kWh"]])


## Figure 4 — Curtailment per response mode + Lorenz curves

Panel A is new (a two-bar total comparison); panels B and C are
`se_plots.plot_curtailment_lorenz` called against the same `concentration` /
`vw_concentration` frames notebook 06 already builds (cells 10-11 and
24-25), passed an external `ax` so both curves sit in one figure instead of
two separate ones.


In [ ]:
# Lorenz curves get 40% of the width each; the bar chart in the middle gets 20%.
fig, (ax_vv, ax_bar, ax_vw) = plt.subplots(
    1, 3, figsize=(16, 5), dpi=130,
    gridspec_kw={"width_ratios": [0.4, 0.2, 0.4]},
)

totals = {
    "Volt-VAr": concentration.attrs.get("total_kWh", 0.0),
    "Volt-Watt": vw_concentration.attrs.get("total_kWh", 0.0),
}
ax_bar.bar(totals.keys(), totals.values(), color=["#7c3aed", "#4709b2"], width=0.55)
for i, (k, v) in enumerate(totals.items()):
    ax_bar.text(i, v, f"{v:,.0f} kWh", ha="center", va="bottom", fontsize=9)
ax_bar.set_ylabel("Estimated curtailment (kWh)")
ax_bar.set_title("Curtailment by response mode", fontsize=10)
ax_bar.grid(color="#ebebeb", lw=0.5, axis="y")
ax_bar.set_axisbelow(True)

plots.plot_curtailment_lorenz(
    concentration, "Volt-VAr concentration",
    f"{concentration.attrs.get('n_affected_sites', 0):,} affected sites", ax=ax_vv)
plots.plot_curtailment_lorenz(
    vw_concentration, "Volt-Watt concentration",
    f"{vw_concentration.attrs.get('n_affected_sites', 0):,} affected sites", ax=ax_vw)

fig.tight_layout()
fig4_path = FIG_DIR / "fig4_curtailment_and_lorenz.png"
fig.savefig(fig4_path, dpi=150, bbox_inches="tight")
print(f"-> {fig4_path}")
fig


## Saved figures

All four land in `FIG_DIR` (`artefacts/publication_figures/` by default) —
paste them straight into the abstract draft in place of the boxed
placeholders.


In [ ]:
for p in sorted(FIG_DIR.glob("fig*.png")):
    print(p)
